In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/snake_dqn")

if "google.colab" in sys.modules:
    if not REPO.exists():
        !git clone -q https://github.com/Mazako/snake_dqn.git {REPO}
    %cd {REPO}
    !git pull -q
    %pip install -q torchrl==0.14.1 tensordict==0.14.3

In [2]:
import torch

from snake_dqn.q_network import RainbowQNetwork

ModuleNotFoundError: No module named 'snake_dqn'

In [ ]:
from functools import partial

from torchrl.envs import Compose, RewardSum, SerialEnv, StepCounter, TransformedEnv
from torchrl.modules import DistributionalQValueActor

from snake_dqn.env import SnakeEnv

SIZE = 11
SEED = 0
N_ENVS = 8
N_ATOMS = 51
V_MIN, V_MAX = (-12, 40)
DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

torch.manual_seed(SEED)
env = TransformedEnv(
    SerialEnv(N_ENVS, partial(SnakeEnv, SIZE, device=DEVICE)),
    Compose([StepCounter(), RewardSum()]),
)
env.set_seed(SEED)
actor = DistributionalQValueActor(
    RainbowQNetwork(SIZE).to(DEVICE),
    support=torch.linspace(V_MIN, V_MAX, N_ATOMS, device=DEVICE),
    in_keys=["observation"],
    spec=env.action_spec,
)

In [ ]:
from torchrl.objectives import DistributionalDQNLoss, HardUpdate

GAMMA = 0.99

loss_module = DistributionalDQNLoss(actor, gamma=GAMMA, delay_value=True)
update = HardUpdate(loss_module=loss_module, value_network_update_interval=1000)

In [ ]:
from torchrl.data import LazyTensorStorage, TensorDictReplayBuffer
from torchrl.envs import MultiStepTransform, UnsqueezeTransform

N_STEPS = 5

buffer = TensorDictReplayBuffer(
    storage=LazyTensorStorage(100_000, ndim=2, device=DEVICE),
    batch_size=128,
)
buffer.append_transform(MultiStepTransform(n_steps=N_STEPS, gamma=GAMMA))
buffer.append_transform(UnsqueezeTransform(dim=-1, in_keys=["steps_to_next_obs"]))

TensorDictReplayBuffer(
    storage=LazyTensorStorage(
        data=<empty>, 
        shape=None, 
        len=0, 
        max_size=100000), 
    sampler=RandomSampler(), 
    writer=TensorDictRoundRobinWriter(cursor=0, full_storage=False), 
Compose(
        MultiStepTransform(keys=[]),
        UnsqueezeTransform(dim=-1, in_keys=['steps_to_next_obs'], out_keys=['steps_to_next_obs'], in_keys_inv=[], out_keys_inv=[])), 
    batch_size=128, 
    collate_fn=<function _collate_id at 0x7003bf3e20>)

In [ ]:
from statistics import mean

from torchrl.collectors import Collector
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import reset_noise

FRAMES_PER_BATCH = 8 * N_ENVS
TOTAL_FRAMES = 50_000
WARMUP_FRAMES = 1_024
UPDATES_PER_BATCH = FRAMES_PER_BATCH // 4
LOG_EVERY = 50

collector = Collector(
    env,
    actor,
    frames_per_batch=FRAMES_PER_BATCH,
    total_frames=TOTAL_FRAMES,
    init_random_frames=WARMUP_FRAMES,
    auto_register_policy_transforms=True,
)
optimizer = torch.optim.Adam(loss_module.parameters(), lr=1e-4)

episode_rewards: list[float] = []
episode_scores: list[int] = []
episode_lengths: list[int] = []
losses: list[float] = []

for i, batch in enumerate(collector):
    done = batch["next", "done"].squeeze(-1)
    episode_rewards.extend(batch["next", "episode_reward"][done].squeeze(-1).tolist())
    episode_lengths.extend(batch["next", "step_count"][done].squeeze(-1).tolist())
    episode_scores.extend(batch["next", "score"][done].squeeze(-1).tolist())
    buffer.extend(batch)

    if len(buffer) >= WARMUP_FRAMES:
        for _ in range(UPDATES_PER_BATCH):
            sample = buffer.sample()
            with set_exploration_type(ExplorationType.RANDOM):
                loss = loss_module(sample)["loss"]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            update.step()
            actor.apply(reset_noise)
            losses.append(loss.item())

    if i % LOG_EVERY == 0 and episode_rewards:
        print(
            f"frames={(i + 1) * FRAMES_PER_BATCH:>7} "
            f"episodes={len(episode_rewards):>5} "
            f"reward={mean(episode_rewards[-50:]):7.2f} "
            f"length={mean(episode_lengths[-50:]):6.1f} "
            f"score={mean(episode_scores[-50:]):5.2f} "
            f"loss={mean(losses[-100:]) if losses else float('nan'):.4f}"
        )

collector.shutdown()

2026-11-08 17:53:21,742 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([12500, 8]) shape [END]


/opt/homebrew/Caskroom/miniforge/base/envs/geron_part_2/lib/python3.12/site-packages/torchrl/collectors/_single.py:1263: UserWarning: total_frames (50000) is not exactly divisible by frames_per_batch (64). This means 48 additional frames will be collected.To silence this message, set the environment variable RL_WARNINGS to False.
  warnings.warn(
